# Deep Exploratory Data Analysis

Electricity spot price forecasting: FR + UK | Metric: RMSE

**Objectives:**
- Understand FR and UK price dynamics in depth
- Identify the key drivers and their interactions
- Detect patterns that are useful for feature engineering
- Quantify the train -> test distribution shift

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.gridspec import GridSpec

from src.data_loading import load_data, merge_train

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

x_train, y_train, x_test = load_data()
train = merge_train(x_train, y_train)

# Forward-fill daily columns for analysis
DAILY_COLS = ['de_gas', 'es_gas', 'fr_gas', 'nl_gas', 'uk_gas', 'eu_emission', 'uk_emission']
for col in DAILY_COLS:
    train[col] = train[col].ffill()
    x_test[col] = x_test[col].ffill()

dt = train['datetime_CET']
dt_test = x_test['datetime_CET']

print(f"Train: {len(train):,} rows × {train.shape[1]} cols")
print(f"Test:  {len(x_test):,} rows")
print(f"Train: {dt.min()} → {dt.max()}")
print(f"Test:  {dt_test.min()} → {dt_test.max()}")
print(f"\nTargets:")
print(y_train[['fr_spot', 'uk_spot']].describe().round(2))

---
## 1. Time Series — Overview of prices

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)

# Prix bruts
axes[0].plot(dt, train['fr_spot'], linewidth=0.3, alpha=0.6, label='FR spot', color='#2166ac')
axes[0].plot(dt, train['uk_spot'], linewidth=0.3, alpha=0.6, label='UK spot', color='#b2182b')
axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_ylabel('EUR/MWh')
axes[0].set_title('Prix spot horaires bruts')
axes[0].legend(loc='upper right')

# Rolling 7 jours
fr_7d = train['fr_spot'].rolling(168).mean()
uk_7d = train['uk_spot'].rolling(168).mean()
axes[1].plot(dt, fr_7d, linewidth=1.5, label='FR (7j rolling)', color='#2166ac')
axes[1].plot(dt, uk_7d, linewidth=1.5, label='UK (7j rolling)', color='#b2182b')
axes[1].fill_between(dt, fr_7d, uk_7d, alpha=0.15, color='gray', label='Spread FR-UK')
axes[1].set_ylabel('EUR/MWh')
axes[1].set_title('Moyenne glissante 7 jours + spread FR-UK')
axes[1].legend(loc='upper right')

# Volatilité (rolling std 24h)
fr_vol = train['fr_spot'].rolling(24).std()
uk_vol = train['uk_spot'].rolling(24).std()
axes[2].plot(dt, fr_vol, linewidth=0.5, alpha=0.7, label='FR volatilité 24h', color='#2166ac')
axes[2].plot(dt, uk_vol, linewidth=0.5, alpha=0.7, label='UK volatilité 24h', color='#b2182b')
axes[2].set_ylabel('Std (EUR/MWh)')
axes[2].set_title('Volatilité intra-journalière (rolling std 24h)')
axes[2].legend(loc='upper right')

# Période test en gris
for ax in axes:
    ax.axvspan(pd.Timestamp('2024-07-01'), pd.Timestamp('2025-02-28'), alpha=0.08, color='green', label='Test period' if ax == axes[0] else None)

plt.tight_layout()
plt.show()

# Stats clés
print(f"Naive 24h RMSE — FR: {np.sqrt(((train['fr_spot'] - train['fr_spot_la'])**2).mean()):.2f}")
print(f"Naive 24h RMSE — UK: {np.sqrt(((train['uk_spot'] - train['uk_spot_la'])**2).mean()):.2f}")
print(f"\nPrix négatifs — FR: {(train['fr_spot']<0).sum()} ({100*(train['fr_spot']<0).mean():.2f}%)")
print(f"Prix négatifs — UK: {(train['uk_spot']<0).sum()} ({100*(train['uk_spot']<0).mean():.2f}%)")
print(f"\nSkewness — FR: {train['fr_spot'].skew():.2f}, UK: {train['uk_spot'].skew():.2f}")
print(f"Kurtosis — FR: {train['fr_spot'].kurtosis():.2f}, UK: {train['uk_spot'].kurtosis():.2f}")

---
## 2. Zoom on typical weeks

In [ ]:
# 4 semaines contrastées : hiver pointe, été creux, semaine avec spike, transition
weeks = [
    ('2023-01-09', '2023-01-15', 'Hiver — semaine froide'),
    ('2023-07-17', '2023-07-23', 'Été — semaine caniculaire'),
    ('2022-08-15', '2022-08-21', 'Crise énergétique 2022'),
    ('2024-04-01', '2024-04-07', 'Printemps — transition'),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for i, (start, end, title) in enumerate(weeks):
    mask = (dt >= start) & (dt <= end)
    sub = train[mask]
    ax = axes[i]
    ax.plot(sub['datetime_CET'], sub['fr_spot'], label='FR', linewidth=1.2, color='#2166ac')
    ax.plot(sub['datetime_CET'], sub['uk_spot'], label='UK', linewidth=1.2, color='#b2182b')
    ax.axhline(0, color='black', linewidth=0.3, linestyle='--')
    ax.set_title(title)
    ax.set_ylabel('EUR/MWh')
    ax.legend(loc='upper right', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d'))

plt.suptitle('Zoom sur 4 semaines typiques', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Detailed daily profiles

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs = GridSpec(3, 2, figure=fig, hspace=0.35)

# Profil horaire moyen par saison
seasons = {
    'Hiver (DJF)': [12, 1, 2],
    'Printemps (MAM)': [3, 4, 5],
    'Été (JJA)': [6, 7, 8],
    'Automne (SON)': [9, 10, 11],
}
colors_season = ['#2166ac', '#4dac26', '#d73027', '#f4a582']

for idx, target in enumerate(['fr_spot', 'uk_spot']):
    ax = fig.add_subplot(gs[0, idx])
    for (season_name, months), color in zip(seasons.items(), colors_season):
        mask = dt.dt.month.isin(months)
        hourly = train.loc[mask].groupby(dt[mask].dt.hour)[target].mean()
        ax.plot(hourly.index, hourly.values, label=season_name, linewidth=2, color=color)
    ax.set_title(f'{target} — Profil horaire par saison')
    ax.set_xlabel('Heure')
    ax.set_ylabel('EUR/MWh')
    ax.legend(fontsize=8)
    ax.set_xticks(range(0, 24))

# Weekday vs Weekend par heure
for idx, target in enumerate(['fr_spot', 'uk_spot']):
    ax = fig.add_subplot(gs[1, idx])
    for is_we, label, color in [(False, 'Semaine', '#2166ac'), (True, 'Weekend', '#d73027')]:
        mask = (dt.dt.dayofweek >= 5) == is_we
        hourly = train.loc[mask].groupby(dt[mask].dt.hour)[target].mean()
        hourly_q25 = train.loc[mask].groupby(dt[mask].dt.hour)[target].quantile(0.25)
        hourly_q75 = train.loc[mask].groupby(dt[mask].dt.hour)[target].quantile(0.75)
        ax.plot(hourly.index, hourly.values, label=label, linewidth=2, color=color)
        ax.fill_between(hourly.index, hourly_q25, hourly_q75, alpha=0.15, color=color)
    ax.set_title(f'{target} — Semaine vs Weekend (bande = IQR)')
    ax.set_xlabel('Heure')
    ax.set_ylabel('EUR/MWh')
    ax.legend(fontsize=8)
    ax.set_xticks(range(0, 24))

# Heatmap heure × mois
for idx, target in enumerate(['fr_spot', 'uk_spot']):
    ax = fig.add_subplot(gs[2, idx])
    pivot = train.assign(hour=dt.dt.hour, month=dt.dt.month).pivot_table(
        values=target, index='hour', columns='month', aggfunc='mean'
    )
    sns.heatmap(pivot, ax=ax, cmap='RdYlBu_r', center=pivot.values.mean(), 
                fmt='.0f', annot=True, annot_kws={'size': 7})
    ax.set_title(f'{target} — Heatmap Heure × Mois (prix moyen)')
    ax.set_ylabel('Heure')
    ax.set_xlabel('Mois')

plt.show()

---
## 4. Autocorrelation and persistence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for idx, target in enumerate(['fr_spot', 'uk_spot']):
    s = train[target].values
    max_lag = 168 * 2  # 2 semaines
    acf = [np.corrcoef(s[lag:], s[:-lag])[0, 1] for lag in range(1, max_lag + 1)]
    
    ax = axes[idx]
    ax.bar(range(1, max_lag + 1), acf, width=1, color='steelblue', alpha=0.7)
    
    # Marquer les lags importants
    for lag, label, color in [(24, '24h', 'red'), (48, '48h', 'orange'), 
                              (168, '7j', 'green'), (336, '14j', 'purple')]:
        if lag <= max_lag:
            ax.axvline(lag, color=color, linestyle='--', linewidth=1, label=f'{label} (r={acf[lag-1]:.3f})')
    
    ax.set_title(f'{target} — Autocorrélation (lags 1h à 2 semaines)')
    ax.set_xlabel('Lag (heures)')
    ax.set_ylabel('Corrélation')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Tableau RMSE par type de naive
print("\nRMSE des baselines naïves :")
print(f"{'Méthode':<30} {'FR RMSE':>10} {'UK RMSE':>10}")
print("-" * 52)
for lag, name in [(1, 'Persistence 1h (t-1)'), (24, 'Persistence 24h (t-24)'),
                  (168, 'Persistence 7j (t-168)')]:
    for target in ['fr_spot', 'uk_spot']:
        shifted = train[target].shift(lag)
        mask = shifted.notna()
        rmse = np.sqrt(((train.loc[mask, target] - shifted[mask])**2).mean())
        if target == 'fr_spot':
            fr_rmse = rmse
        else:
            print(f"{name:<30} {fr_rmse:>10.2f} {rmse:>10.2f}")

---
## 5. Missing Data — Detailed analysis

In [ ]:
# NaN par colonne
nan_train = x_train.drop(columns=['datetime_CET', 'datetime_UTC']).isnull().mean() * 100
nan_test = x_test.drop(columns=['datetime_CET', 'datetime_UTC']).isnull().mean() * 100

nan_df = pd.DataFrame({'Train %': nan_train, 'Test %': nan_test}).sort_values('Train %', ascending=False)
nan_df = nan_df[nan_df['Train %'] > 0]

fig, ax = plt.subplots(figsize=(12, max(6, len(nan_df) * 0.3)))
y_pos = range(len(nan_df))
ax.barh(y_pos, nan_df['Train %'], height=0.4, label='Train', color='steelblue', align='center')
ax.barh([y + 0.4 for y in y_pos], nan_df['Test %'], height=0.4, label='Test', color='coral', align='center')
ax.set_yticks([y + 0.2 for y in y_pos])
ax.set_yticklabels(nan_df.index, fontsize=8)
ax.set_xlabel('Missing %')
ax.set_title('Missing data : Train vs Test')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Quand les données apparaissent (timeline)
print("\nPremière valeur non-NaN par colonne sparse (Train) :")
sparse_cols = nan_train[nan_train > 5].index.tolist()
for col in sparse_cols[:15]:
    first_valid = x_train[col].first_valid_index()
    if first_valid is not None:
        first_date = x_train.loc[first_valid, 'datetime_CET']
        pct_avail = 100 * x_train[col].notna().mean()
        print(f"  {col:<35} Début: {first_date}  ({pct_avail:.1f}% dispo)")

---
## 6. Top correlations + targeted correlation matrix

In [ ]:
numeric_cols = x_train.select_dtypes(include=[np.number]).columns.tolist()

# Top 25 pour chaque cible
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for idx, target in enumerate(['fr_spot', 'uk_spot']):
    corr = x_train[numeric_cols].corrwith(y_train[target]).dropna()
    top25 = corr.abs().sort_values(ascending=True).tail(25)
    colors = ['#d73027' if corr[c] < 0 else '#2166ac' for c in top25.index]
    axes[idx].barh(range(len(top25)), [corr[c] for c in top25.index], color=colors)
    axes[idx].set_yticks(range(len(top25)))
    axes[idx].set_yticklabels(top25.index, fontsize=8)
    axes[idx].set_title(f'Top 25 corrélations avec {target} (signé)')
    axes[idx].set_xlabel('Pearson r')
    axes[idx].axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

# Matrice ciblée : features les plus corrélées + targets
key_feats = ['fr_spot_la', 'uk_spot_la', 'de_spot_la', 'be_spot_la', 'nl_spot_la',
             'fr_load_f', 'uk_load_f', 'de_load_f',
             'fr_wind_f', 'uk_wind_f', 'fr_solar_f', 'uk_solar_f',
             'fr_nuclear_avcap_f', 'uk_nuclear_avcap_f', 'uk_gas_avcap_f',
             'fr_gas', 'uk_gas', 'eu_emission']
key_feats = [f for f in key_feats if f in train.columns]

corr_matrix = train[key_feats + ['fr_spot', 'uk_spot']].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, annot_kws={'size': 8}, vmin=-1, vmax=1)
ax.set_title('Matrice de corrélation — Features clés vs Targets')
plt.tight_layout()
plt.show()

---
## 7. Merit Order — Residual Load vs Price

In [ ]:
# Créer residual loads
train['fr_residual'] = train['fr_load_f'] - train['fr_solar_f'] - train['fr_wind_f']
train['uk_residual'] = train['uk_load_f'] - train['uk_solar_f'] - train['uk_wind_f']
train['de_residual'] = train['de_load_f'] - train['de_solar_f'] - train['de_wind_f']
train['fr_thermal_need'] = train['fr_residual'] - train['fr_nuclear_avcap_f']
train['uk_thermal_need'] = train['uk_residual'] - train['uk_nuclear_avcap_f']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# FR : residual load vs price
sc = axes[0, 0].scatter(train['fr_residual'], train['fr_spot'], 
                         c=dt.dt.month, cmap='hsv', s=1, alpha=0.3)
axes[0, 0].set_xlabel('FR Residual Load (MW)')
axes[0, 0].set_ylabel('FR Spot (EUR/MWh)')
axes[0, 0].set_title('FR: Residual Load vs Prix (couleur = mois)')
plt.colorbar(sc, ax=axes[0, 0], label='Mois')

# FR : thermal need vs price (ce que le gaz/charbon doit couvrir)
axes[0, 1].scatter(train['fr_thermal_need'], train['fr_spot'], s=1, alpha=0.3, color='#d73027')
axes[0, 1].set_xlabel('FR Thermal Need (MW) = Residual - Nuclear')
axes[0, 1].set_ylabel('FR Spot')
axes[0, 1].set_title('FR: Besoin thermique vs Prix')
axes[0, 1].axvline(0, color='black', linestyle='--', linewidth=0.5)

# FR : nuclear capacity vs price
axes[0, 2].scatter(train['fr_nuclear_avcap_f'], train['fr_spot'], s=1, alpha=0.3, color='#2166ac')
axes[0, 2].set_xlabel('FR Nuclear Available Capacity (MW)')
axes[0, 2].set_ylabel('FR Spot')
axes[0, 2].set_title('FR: Capacité nucléaire vs Prix')

# UK : residual load vs price
sc = axes[1, 0].scatter(train['uk_residual'], train['uk_spot'],
                         c=dt.dt.month, cmap='hsv', s=1, alpha=0.3)
axes[1, 0].set_xlabel('UK Residual Load (MW)')
axes[1, 0].set_ylabel('UK Spot (EUR/MWh)')
axes[1, 0].set_title('UK: Residual Load vs Prix (couleur = mois)')
plt.colorbar(sc, ax=axes[1, 0], label='Mois')

# UK : wind vs price (UK est très éolien)
axes[1, 1].scatter(train['uk_wind_f'], train['uk_spot'], s=1, alpha=0.3, color='#4dac26')
axes[1, 1].set_xlabel('UK Wind Forecast (MW)')
axes[1, 1].set_ylabel('UK Spot')
axes[1, 1].set_title('UK: Éolien vs Prix (merit order effect)')

# UK : gas capacity vs price
axes[1, 2].scatter(train['uk_gas_avcap_f'], train['uk_spot'], s=1, alpha=0.3, color='#b2182b')
axes[1, 2].set_xlabel('UK Gas Available Capacity (MW)')
axes[1, 2].set_ylabel('UK Spot')
axes[1, 2].set_title('UK: Capacité gaz vs Prix')

plt.tight_layout()
plt.show()

print(f"Corrélations avec fr_spot :")
print(f"  fr_residual:      {train['fr_residual'].corr(train['fr_spot']):.3f}")
print(f"  fr_thermal_need:  {train['fr_thermal_need'].corr(train['fr_spot']):.3f}")
print(f"  fr_nuclear_avcap: {train['fr_nuclear_avcap_f'].corr(train['fr_spot']):.3f}")
print(f"\nCorrélations avec uk_spot :")
print(f"  uk_residual:      {train['uk_residual'].corr(train['uk_spot']):.3f}")
print(f"  uk_wind_f:        {train['uk_wind_f'].corr(train['uk_spot']):.3f}")
print(f"  uk_gas_avcap_f:   {train['uk_gas_avcap_f'].corr(train['uk_spot']):.3f}")

---
## 8. Commodities — Gas and Carbon as price drivers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Time series gaz + émissions
ax = axes[0, 0]
ax.plot(dt, train['fr_gas'], label='FR gas (PEG)', linewidth=1)
ax.plot(dt, train['uk_gas'], label='UK gas (NBP)', linewidth=1)
ax.plot(dt, train['nl_gas'], label='NL gas (TTF)', linewidth=1)
ax.set_title('Prix du gaz au fil du temps')
ax.set_ylabel('EUR/MWh')
ax.legend(fontsize=8)

ax = axes[0, 1]
ax.plot(dt, train['eu_emission'], label='EU ETS (EUA)', linewidth=1, color='#2166ac')
ax.plot(dt, train['uk_emission'], label='UK ETS (UKA)', linewidth=1, color='#b2182b')
ax.set_title('Prix du carbone au fil du temps')
ax.set_ylabel('EUR/tCO2')
ax.legend(fontsize=8)

# Spark spread approximation
GAS_EFF = 0.50
EMISSION_FACTOR = 0.37
train['fr_spark'] = train['fr_gas'] / GAS_EFF + train['eu_emission'] * EMISSION_FACTOR
train['uk_spark'] = train['uk_gas'] / GAS_EFF + train['uk_emission'] * EMISSION_FACTOR

# FR spark vs FR price
mask_spark = train['fr_spark'].notna()
axes[1, 0].scatter(train.loc[mask_spark, 'fr_spark'], train.loc[mask_spark, 'fr_spot'], 
                   s=2, alpha=0.3, color='#2166ac')
axes[1, 0].plot([0, 400], [0, 400], 'r--', linewidth=1, label='y=x')
axes[1, 0].set_xlabel('FR Clean Spark Spread (EUR/MWh)')
axes[1, 0].set_ylabel('FR Spot')
axes[1, 0].set_title('FR: Coût marginal gaz vs Prix spot')
axes[1, 0].legend()

mask_spark = train['uk_spark'].notna()
axes[1, 1].scatter(train.loc[mask_spark, 'uk_spark'], train.loc[mask_spark, 'uk_spot'],
                   s=2, alpha=0.3, color='#b2182b')
axes[1, 1].plot([0, 400], [0, 400], 'r--', linewidth=1, label='y=x')
axes[1, 1].set_xlabel('UK Clean Spark Spread (EUR/MWh)')
axes[1, 1].set_ylabel('UK Spot')
axes[1, 1].set_title('UK: Coût marginal gaz vs Prix spot')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"Corrélation spark spread vs prix spot :")
print(f"  FR: {train['fr_spark'].corr(train['fr_spot']):.3f}")
print(f"  UK: {train['uk_spark'].corr(train['uk_spot']):.3f}")

---
## 9. UK Interconnectors — Flows, Capacity, Congestion

In [ ]:
# Flux nets laggés sur les 6 interconnecteurs UK
interconnectors = {
    'IFA1 (FR-UK)':    ('flow_fr-uk-1_la', 'flow_uk-fr-1_la'),
    'IFA2 (FR-UK)':    ('flow_fr-uk-2_la', 'flow_uk-fr-2_la'),
    'ElecLink (FR-UK)':('flow_fr-uk-3_la', 'flow_uk-fr-3_la'),
    'NEMO (BE-UK)':    ('flow_be-uk_la',   'flow_uk-be_la'),
    'BritNed (NL-UK)': ('flow_nl-uk_la',   'flow_uk-nl_la'),
    'Viking (DK1-UK)': ('flow_dk1-uk_la',  'flow_uk-dk1_la'),
}

fig, axes = plt.subplots(3, 2, figsize=(18, 14))
axes = axes.flatten()

for i, (name, (to_uk, from_uk)) in enumerate(interconnectors.items()):
    net = train[to_uk].fillna(0) - train[from_uk].fillna(0)
    ax = axes[i]
    ax.plot(dt, net.rolling(168).mean(), linewidth=1, color='steelblue')
    ax.fill_between(dt, 0, net.rolling(168).mean(), alpha=0.2, color='steelblue')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{name} — Flux net vers UK (rolling 7j)')
    ax.set_ylabel('MW (>0 = import UK)')

plt.suptitle('Flux nets sur les 6 interconnecteurs UK (lagged 24h)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Total des flux et relation avec prix
flow_to_uk_cols = ['flow_be-uk_la', 'flow_dk1-uk_la', 'flow_fr-uk-1_la', 
                   'flow_fr-uk-2_la', 'flow_fr-uk-3_la', 'flow_nl-uk_la']
flow_from_uk_cols = ['flow_uk-be_la', 'flow_uk-dk1_la', 'flow_uk-fr-1_la',
                     'flow_uk-fr-2_la', 'flow_uk-fr-3_la', 'flow_uk-nl_la']

train['net_import_uk'] = train[flow_to_uk_cols].sum(axis=1) - train[flow_from_uk_cols].sum(axis=1)
train['fr_uk_net_flow'] = (train[['flow_fr-uk-1_la', 'flow_fr-uk-2_la', 'flow_fr-uk-3_la']].sum(axis=1) 
                          - train[['flow_uk-fr-1_la', 'flow_uk-fr-2_la', 'flow_uk-fr-3_la']].sum(axis=1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(train['net_import_uk'], train['uk_spot'], s=1, alpha=0.2)
axes[0].set_xlabel('UK Net Import total (MW, lagged)')
axes[0].set_ylabel('UK Spot')
axes[0].set_title(f'UK: Import net vs Prix (r={train["net_import_uk"].corr(train["uk_spot"]):.3f})')

axes[1].scatter(train['fr_uk_net_flow'], train['uk_spot'], s=1, alpha=0.2, color='#b2182b')
axes[1].set_xlabel('FR→UK Net Flow (MW, lagged)')
axes[1].set_ylabel('UK Spot')
axes[1].set_title(f'FR→UK flow vs UK prix (r={train["fr_uk_net_flow"].corr(train["uk_spot"]):.3f})')

axes[2].scatter(train['fr_uk_net_flow'], train['fr_spot'], s=1, alpha=0.2, color='#2166ac')
axes[2].set_xlabel('FR→UK Net Flow (MW, lagged)')
axes[2].set_ylabel('FR Spot')
axes[2].set_title(f'FR→UK flow vs FR prix (r={train["fr_uk_net_flow"].corr(train["fr_spot"]):.3f})')

plt.tight_layout()
plt.show()

---
## 10. FR-UK Spread — Coupling dynamics

In [ ]:
train['spread_fr_uk'] = train['fr_spot'] - train['uk_spot']

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Time series du spread
axes[0, 0].plot(dt, train['spread_fr_uk'].rolling(168).mean(), linewidth=1, color='purple')
axes[0, 0].fill_between(dt, 0, train['spread_fr_uk'].rolling(168).mean(), alpha=0.2, color='purple')
axes[0, 0].axhline(0, color='black', linewidth=0.5)
axes[0, 0].set_title('Spread FR - UK (rolling 7j)')
axes[0, 0].set_ylabel('EUR/MWh (>0 = FR plus cher)')

# Distribution du spread
axes[0, 1].hist(train['spread_fr_uk'], bins=100, edgecolor='black', alpha=0.7, color='purple')
axes[0, 1].axvline(train['spread_fr_uk'].mean(), color='red', linestyle='--', 
                   label=f'Mean={train["spread_fr_uk"].mean():.1f}')
axes[0, 1].set_title('Distribution du spread FR - UK')
axes[0, 1].set_xlabel('EUR/MWh')
axes[0, 1].legend()

# Spread par heure
spread_hourly = train.groupby(dt.dt.hour)['spread_fr_uk'].agg(['mean', 'std'])
axes[1, 0].bar(spread_hourly.index, spread_hourly['mean'], color='purple', alpha=0.7)
axes[1, 0].errorbar(spread_hourly.index, spread_hourly['mean'], yerr=spread_hourly['std'],
                    fmt='none', color='black', capsize=2)
axes[1, 0].axhline(0, color='black', linewidth=0.5)
axes[1, 0].set_title('Spread FR-UK par heure')
axes[1, 0].set_xlabel('Heure')
axes[1, 0].set_ylabel('EUR/MWh')

# Spread vs FR-UK ATC total (quand la capacité est pleine, les prix divergent)
atc_fr_uk = train[['atc_fr-uk-1_f', 'atc_fr-uk-2_f', 'atc_fr-uk-3_f']].sum(axis=1)
axes[1, 1].scatter(atc_fr_uk, train['spread_fr_uk'].abs(), s=1, alpha=0.2)
axes[1, 1].set_xlabel('ATC FR→UK total (MW)')
axes[1, 1].set_ylabel('|Spread FR-UK| (EUR/MWh)')
axes[1, 1].set_title('Capacité interconnexion vs Divergence de prix')

plt.tight_layout()
plt.show()

print(f"\nSpread FR-UK stats:")
print(f"  Mean:   {train['spread_fr_uk'].mean():.2f} EUR/MWh")
print(f"  Std:    {train['spread_fr_uk'].std():.2f}")
print(f"  FR>UK:  {(train['spread_fr_uk'] > 0).mean()*100:.1f}% du temps")

---
## 11. Spikes — In-depth analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for row, target in enumerate(['fr_spot', 'uk_spot']):
    prefix = target.split('_')[0]
    p99 = y_train[target].quantile(0.99)
    p01 = y_train[target].quantile(0.01)
    is_spike = train[target] > p99
    is_neg = train[target] < p01
    
    # Timeline des spikes
    ax = axes[row, 0]
    ax.plot(dt, train[target], linewidth=0.2, alpha=0.5, color='gray')
    ax.scatter(dt[is_spike], train.loc[is_spike, target], s=10, color='red', zorder=5, label=f'>p99 ({p99:.0f})')
    ax.scatter(dt[is_neg], train.loc[is_neg, target], s=10, color='blue', zorder=5, label=f'<p01 ({p01:.0f})')
    ax.set_title(f'{target} — Spikes et prix négatifs')
    ax.set_ylabel('EUR/MWh')
    ax.legend(fontsize=8)
    
    # Conditions pendant les spikes
    ax = axes[row, 1]
    if prefix == 'fr':
        features = ['fr_nuclear_avcap_f', 'fr_wind_f', 'fr_load_f', 'fr_residual']
    else:
        features = ['uk_gas_avcap_f', 'uk_wind_f', 'uk_load_f', 'uk_residual']
    
    spike_data = train.loc[is_spike, features].mean()
    normal_data = train.loc[~is_spike, features].mean()
    ratio = (spike_data / normal_data - 1) * 100
    
    colors = ['#d73027' if v < 0 else '#2166ac' for v in ratio]
    ax.barh(range(len(ratio)), ratio.values, color=colors)
    ax.set_yticks(range(len(ratio)))
    ax.set_yticklabels(ratio.index, fontsize=9)
    ax.set_xlabel('% de différence vs normal')
    ax.set_title(f'{target} — Conditions pendant les spikes')
    ax.axvline(0, color='black', linewidth=0.5)
    
    # Heatmap heure × mois pour les spikes
    ax = axes[row, 2]
    spike_counts = train[is_spike].assign(
        hour=dt[is_spike].dt.hour, month=dt[is_spike].dt.month
    ).pivot_table(values=target, index='hour', columns='month', aggfunc='count', fill_value=0)
    # Reindex to ensure all hours and months
    spike_counts = spike_counts.reindex(index=range(24), columns=range(1, 13), fill_value=0)
    sns.heatmap(spike_counts, ax=ax, cmap='Reds', annot=True, fmt='d', annot_kws={'size': 7})
    ax.set_title(f'{target} — Concentration des spikes (Heure × Mois)')
    ax.set_ylabel('Heure')
    ax.set_xlabel('Mois')

plt.tight_layout()
plt.show()

---
## 12. Wind and Solar — Impact on price formation

In [ ]:
# Wind penetration vs price
train['fr_wind_pen'] = train['fr_wind_f'] / train['fr_load_f'].clip(lower=1)
train['uk_wind_pen'] = train['uk_wind_f'] / train['uk_load_f'].clip(lower=1)
train['fr_solar_pen'] = train['fr_solar_f'] / train['fr_load_f'].clip(lower=1)
train['fr_renewable_pen'] = (train['fr_wind_f'] + train['fr_solar_f']) / train['fr_load_f'].clip(lower=1)
train['uk_renewable_pen'] = (train['uk_wind_f'] + train['uk_solar_f']) / train['uk_load_f'].clip(lower=1)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# FR
axes[0, 0].scatter(train['fr_wind_pen']*100, train['fr_spot'], s=1, alpha=0.2, color='#4dac26')
# Binned average
bins = pd.cut(train['fr_wind_pen']*100, bins=20)
binned = train.groupby(bins, observed=True)['fr_spot'].agg(['mean', 'median'])
bin_centers = [(b.left + b.right)/2 for b in binned.index]
axes[0, 0].plot(bin_centers, binned['mean'], 'r-', linewidth=2, label='Moyenne par bin')
axes[0, 0].plot(bin_centers, binned['median'], 'k--', linewidth=2, label='Médiane par bin')
axes[0, 0].set_xlabel('FR Wind Penetration (%)')
axes[0, 0].set_ylabel('FR Spot')
axes[0, 0].set_title('FR: Pénétration éolienne vs Prix')
axes[0, 0].legend(fontsize=8)

axes[0, 1].scatter(train['fr_solar_pen']*100, train['fr_spot'], s=1, alpha=0.2, color='#ff7f00')
bins = pd.cut(train['fr_solar_pen']*100, bins=20)
binned = train.groupby(bins, observed=True)['fr_spot'].agg(['mean', 'median'])
bin_centers = [(b.left + b.right)/2 for b in binned.index]
axes[0, 1].plot(bin_centers, binned['mean'], 'r-', linewidth=2, label='Moyenne par bin')
axes[0, 1].set_xlabel('FR Solar Penetration (%)')
axes[0, 1].set_ylabel('FR Spot')
axes[0, 1].set_title('FR: Pénétration solaire vs Prix')
axes[0, 1].legend(fontsize=8)

# FR renewable total
axes[0, 2].scatter(train['fr_renewable_pen']*100, train['fr_spot'], s=1, alpha=0.2, color='#1b7837')
bins = pd.cut(train['fr_renewable_pen']*100, bins=20)
binned = train.groupby(bins, observed=True)['fr_spot'].agg(['mean', 'median'])
bin_centers = [(b.left + b.right)/2 for b in binned.index]
axes[0, 2].plot(bin_centers, binned['mean'], 'r-', linewidth=2, label='Moyenne par bin')
axes[0, 2].set_xlabel('FR Renewable Penetration (%)')
axes[0, 2].set_ylabel('FR Spot')
axes[0, 2].set_title('FR: Pénétration ENR totale vs Prix')
axes[0, 2].legend(fontsize=8)

# UK
axes[1, 0].scatter(train['uk_wind_pen']*100, train['uk_spot'], s=1, alpha=0.2, color='#4dac26')
bins = pd.cut(train['uk_wind_pen']*100, bins=20)
binned = train.groupby(bins, observed=True)['uk_spot'].agg(['mean', 'median'])
bin_centers = [(b.left + b.right)/2 for b in binned.index]
axes[1, 0].plot(bin_centers, binned['mean'], 'r-', linewidth=2, label='Moyenne par bin')
axes[1, 0].plot(bin_centers, binned['median'], 'k--', linewidth=2, label='Médiane par bin')
axes[1, 0].set_xlabel('UK Wind Penetration (%)')
axes[1, 0].set_ylabel('UK Spot')
axes[1, 0].set_title('UK: Pénétration éolienne vs Prix')
axes[1, 0].legend(fontsize=8)

# Wind time series + rolling
axes[1, 1].plot(dt, train['uk_wind_f'].rolling(168).mean(), linewidth=1, color='#4dac26', label='UK wind (7j avg)')
axes[1, 1].plot(dt, train['fr_wind_f'].rolling(168).mean(), linewidth=1, color='#2166ac', label='FR wind (7j avg)')
ax2 = axes[1, 1].twinx()
ax2.plot(dt, train['uk_spot'].rolling(168).mean(), linewidth=1, color='#b2182b', alpha=0.5, label='UK price (7j avg)')
ax2.set_ylabel('UK Spot (EUR/MWh)', color='#b2182b')
axes[1, 1].set_ylabel('Wind (MW)')
axes[1, 1].set_title('Éolien vs Prix au fil du temps')
axes[1, 1].legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)

# Renewable penetration UK
axes[1, 2].scatter(train['uk_renewable_pen']*100, train['uk_spot'], s=1, alpha=0.2, color='#1b7837')
bins = pd.cut(train['uk_renewable_pen']*100, bins=20)
binned = train.groupby(bins, observed=True)['uk_spot'].agg(['mean', 'median'])
bin_centers = [(b.left + b.right)/2 for b in binned.index]
axes[1, 2].plot(bin_centers, binned['mean'], 'r-', linewidth=2, label='Moyenne par bin')
axes[1, 2].set_xlabel('UK Renewable Penetration (%)')
axes[1, 2].set_ylabel('UK Spot')
axes[1, 2].set_title('UK: Pénétration ENR totale vs Prix')
axes[1, 2].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 13. FR Nuclear — 2022 crisis and recovery

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Nuclear capacity time series
ax = axes[0, 0]
ax.plot(dt, train['fr_nuclear_avcap_f'], linewidth=0.5, alpha=0.7, color='#2166ac')
ax.plot(dt, train['fr_nuclear_avcap_f'].rolling(168).mean(), linewidth=2, color='darkblue', label='7j avg')
ax.axhline(train['fr_nuclear_avcap_f'].median(), color='red', linestyle='--', linewidth=1,
           label=f'Médiane={train["fr_nuclear_avcap_f"].median():.0f} MW')
ax.set_title('FR Nuclear Available Capacity — Evolution')
ax.set_ylabel('MW')
ax.legend(fontsize=8)

# Nuclear vs price coloré par saison
ax = axes[0, 1]
for (season_name, months), color in zip(seasons.items(), colors_season):
    mask = dt.dt.month.isin(months)
    ax.scatter(train.loc[mask, 'fr_nuclear_avcap_f'], train.loc[mask, 'fr_spot'],
              s=2, alpha=0.3, color=color, label=season_name)
ax.set_xlabel('FR Nuclear Capacity (MW)')
ax.set_ylabel('FR Spot (EUR/MWh)')
ax.set_title('FR: Nucléaire vs Prix par saison')
ax.legend(fontsize=8)

# River temperature vs nuclear capacity
ax = axes[1, 0]
sc = ax.scatter(train['fr_river_temp_rhone_lyon_f'], train['fr_nuclear_avcap_f'],
                c=train['fr_spot'], cmap='RdYlBu_r', s=2, alpha=0.4, vmin=-50, vmax=300)
ax.set_xlabel('Rhône temperature Lyon (°C)')
ax.set_ylabel('FR Nuclear Capacity (MW)')
ax.set_title('Température rivière vs Nucléaire (couleur = prix)')
ax.axvline(25, color='red', linestyle='--', linewidth=1, label='Seuil 25°C')
ax.legend()
plt.colorbar(sc, ax=ax, label='FR Spot (EUR/MWh)')

# River temps over time
ax = axes[1, 1]
ax.plot(dt, train['fr_river_temp_rhone_lyon_f'], linewidth=0.5, alpha=0.7, color='#d73027', label='Rhône Lyon')
ax.plot(dt, train['fr_river_temp_rhine_rheinfelden_f'], linewidth=0.5, alpha=0.7, color='#2166ac', label='Rhin Rheinfelden')
ax.axhline(25, color='red', linestyle='--', linewidth=1, label='Seuil contrainte')
ax.set_title('Températures des rivières (contraintes nucléaire/thermique)')
ax.set_ylabel('°C')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Quantifier l'impact nucléaire
low_nuke = train['fr_nuclear_avcap_f'] < train['fr_nuclear_avcap_f'].quantile(0.1)
high_nuke = train['fr_nuclear_avcap_f'] > train['fr_nuclear_avcap_f'].quantile(0.9)
print(f"FR Spot quand nucléaire bas (p10) : {train.loc[low_nuke, 'fr_spot'].mean():.1f} EUR/MWh")
print(f"FR Spot quand nucléaire haut (p90): {train.loc[high_nuke, 'fr_spot'].mean():.1f} EUR/MWh")
print(f"Différence: {train.loc[low_nuke, 'fr_spot'].mean() - train.loc[high_nuke, 'fr_spot'].mean():.1f} EUR/MWh")

---
## 14. Cross-border price coupling — Neighbor prices vs FR/UK

In [ ]:
spot_la_cols = [c for c in train.columns if c.endswith('_spot_la')]
zones = [c.replace('_spot_la', '').upper() for c in spot_la_cols]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Corrélation de chaque zone lagged avec FR spot
corrs_fr = {z: train[c].corr(train['fr_spot']) for z, c in zip(zones, spot_la_cols)}
corrs_uk = {z: train[c].corr(train['uk_spot']) for z, c in zip(zones, spot_la_cols)}

sorted_fr = sorted(corrs_fr.items(), key=lambda x: x[1], reverse=True)
sorted_uk = sorted(corrs_uk.items(), key=lambda x: x[1], reverse=True)

axes[0].barh([x[0] for x in sorted_fr], [x[1] for x in sorted_fr], color='#2166ac')
axes[0].set_title('Corrélation prix voisins (lag 24h) vs FR spot')
axes[0].set_xlabel('Pearson r')

axes[1].barh([x[0] for x in sorted_uk], [x[1] for x in sorted_uk], color='#b2182b')
axes[1].set_title('Corrélation prix voisins (lag 24h) vs UK spot')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.show()

# Evolution temporelle des corrélations (rolling 30j)
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
window = 720  # 30 jours

for c, z in [('fr_spot_la', 'FR'), ('de_spot_la', 'DE'), ('be_spot_la', 'BE'), ('uk_spot_la', 'UK')]:
    rolling_corr = train['fr_spot'].rolling(window).corr(train[c])
    axes[0].plot(dt, rolling_corr, linewidth=1, label=f'{z} lag', alpha=0.8)
axes[0].set_title('Corrélation glissante 30j — vs FR spot')
axes[0].set_ylabel('Corrélation')
axes[0].legend(fontsize=8)

for c, z in [('uk_spot_la', 'UK'), ('fr_spot_la', 'FR'), ('nl_spot_la', 'NL'), ('be_spot_la', 'BE')]:
    rolling_corr = train['uk_spot'].rolling(window).corr(train[c])
    axes[1].plot(dt, rolling_corr, linewidth=1, label=f'{z} lag', alpha=0.8)
axes[1].set_title('Corrélation glissante 30j — vs UK spot')
axes[1].set_ylabel('Corrélation')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 15. Load — Demand and its patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Load time series
ax = axes[0, 0]
ax.plot(dt, train['fr_load_f'].rolling(168).mean(), linewidth=1, color='#2166ac', label='FR load (7j avg)')
ax.plot(dt, train['uk_load_f'].rolling(168).mean(), linewidth=1, color='#b2182b', label='UK load (7j avg)')
ax.set_title('Demande FR et UK (rolling 7j)')
ax.set_ylabel('MW')
ax.legend(fontsize=8)

# Load vs price
ax = axes[0, 1]
ax.scatter(train['fr_load_f'], train['fr_spot'], s=1, alpha=0.2, color='#2166ac', label='FR')
ax.set_xlabel('FR Load (MW)')
ax.set_ylabel('FR Spot')
ax.set_title(f'FR: Load vs Prix (r={train["fr_load_f"].corr(train["fr_spot"]):.3f})')

# Residual load time series
ax = axes[1, 0]
ax.plot(dt, train['fr_residual'].rolling(168).mean(), linewidth=1, color='#2166ac', label='FR residual')
ax.plot(dt, train['uk_residual'].rolling(168).mean(), linewidth=1, color='#b2182b', label='UK residual')
ax.set_title('Residual Load FR et UK (rolling 7j)')
ax.set_ylabel('MW')
ax.legend(fontsize=8)

# DE load vs FR price (coupling continental)
ax = axes[1, 1]
ax.scatter(train['de_load_f'], train['fr_spot'], s=1, alpha=0.15, color='orange')
ax.set_xlabel('DE Load (MW)')
ax.set_ylabel('FR Spot')
ax.set_title(f'DE Load vs FR Prix (r={train["de_load_f"].corr(train["fr_spot"]):.3f}) — couplage continental')

plt.tight_layout()
plt.show()

---
## 16. Distribution Shift — Train vs Test

In [ ]:
# Features clés : comparer distributions
compare_feats = [
    'fr_load_f', 'uk_load_f', 'fr_wind_f', 'uk_wind_f',
    'fr_solar_f', 'uk_solar_f', 'fr_nuclear_avcap_f', 'uk_nuclear_avcap_f',
    'fr_spot_la', 'uk_spot_la', 'uk_gas_avcap_f', 'de_load_f',
]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(compare_feats):
    ax = axes[i]
    tr = x_train[feat].dropna()
    te = x_test[feat].dropna()
    
    ax.hist(tr, bins=50, alpha=0.5, label=f'Train (μ={tr.mean():.0f})', density=True, color='steelblue')
    ax.hist(te, bins=50, alpha=0.5, label=f'Test (μ={te.mean():.0f})', density=True, color='coral')
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Distribution Shift — Train vs Test', fontsize=14)
plt.tight_layout()
plt.show()

# Quantifier le shift
print("\nDistribution shift (Test mean / Train mean) :")
print(f"{'Feature':<30} {'Train mean':>12} {'Test mean':>12} {'Ratio':>8}")
print("-" * 65)
for feat in compare_feats:
    tr_mean = x_train[feat].mean()
    te_mean = x_test[feat].mean()
    ratio = te_mean / tr_mean if tr_mean != 0 else float('inf')
    flag = ' ⚠' if abs(ratio - 1) > 0.15 else ''
    print(f"{feat:<30} {tr_mean:>12.1f} {te_mean:>12.1f} {ratio:>7.2f}x{flag}")

---
## 17. Hydro — Reservoirs and run-of-river

In [ ]:
hydro_cols = [c for c in train.columns if 'hydro' in c]
print(f"Colonnes hydro : {hydro_cols}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# FR hydro over time
ax = axes[0]
ax.plot(dt, train['fr_hydro_res_f'].rolling(168).mean(), linewidth=1.5, label='FR Hydro Réservoir', color='#2166ac')
ax.plot(dt, train['fr_hydro_ror_f'].rolling(168).mean(), linewidth=1.5, label='FR Hydro Fil de l\'eau', color='#4dac26')
ax.set_title('FR Hydro — Réservoir vs Fil de l\'eau (7j avg)')
ax.set_ylabel('MW')
ax.legend(fontsize=8)

# FR hydro vs price
ax = axes[1]
ax.scatter(train['fr_hydro_res_f'], train['fr_spot'], s=1, alpha=0.2, color='#2166ac')
ax.set_xlabel('FR Hydro Réservoir (MW)')
ax.set_ylabel('FR Spot')
ax.set_title(f'FR: Hydro réservoir vs Prix (r={train["fr_hydro_res_f"].corr(train["fr_spot"]):.3f})')

# CH + AT hydro (important pour prix FR via couplage)
ax = axes[2]
ch_at_hydro = train['ch_hydro_res_f'] + train['at_hydro_res_f']
ax.scatter(ch_at_hydro, train['fr_spot'], s=1, alpha=0.2, color='#762a83')
ax.set_xlabel('CH + AT Hydro Réservoir (MW)')
ax.set_ylabel('FR Spot')
ax.set_title(f'CH+AT Hydro vs FR Prix (r={ch_at_hydro.corr(train["fr_spot"]):.3f})')

plt.tight_layout()
plt.show()

---
## 18. Lagged Prices — Persistence and momentum

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for idx, (target, lag_col) in enumerate([('fr_spot', 'fr_spot_la'), ('uk_spot', 'uk_spot_la')]):
    # Scatter lag vs actual
    ax = axes[idx, 0]
    ax.scatter(train[lag_col], train[target], s=1, alpha=0.2)
    lims = [min(train[lag_col].min(), train[target].min()), 
            max(train[lag_col].max(), train[target].max())]
    ax.plot(lims, lims, 'r--', linewidth=1, label='y=x')
    r = train[lag_col].corr(train[target])
    ax.set_xlabel(f'{lag_col} (t-24h)')
    ax.set_ylabel(f'{target} (t)')
    ax.set_title(f'{target}: Lag 24h vs Actual (r={r:.3f})')
    ax.legend()
    
    # Résidus du naive (erreur = actual - lag)
    ax = axes[idx, 1]
    resid = train[target] - train[lag_col]
    resid_hourly = resid.groupby(dt.dt.hour).agg(['mean', 'std'])
    ax.bar(resid_hourly.index, resid_hourly['mean'], color='steelblue', alpha=0.7)
    ax.errorbar(resid_hourly.index, resid_hourly['mean'], yerr=resid_hourly['std'],
               fmt='none', color='black', capsize=2)
    ax.axhline(0, color='red', linewidth=0.5)
    ax.set_title(f'{target}: Biais du naive 24h par heure')
    ax.set_xlabel('Heure')
    ax.set_ylabel('Erreur moyenne (EUR/MWh)')

plt.tight_layout()
plt.show()

# Erreur du naive par mois
print("\nRMSE du naive 24h par mois :")
for target, lag_col in [('fr_spot', 'fr_spot_la'), ('uk_spot', 'uk_spot_la')]:
    resid = train[target] - train[lag_col]
    monthly_rmse = resid.groupby(dt.dt.month).apply(lambda x: np.sqrt((x**2).mean()))
    print(f"\n  {target}:")
    for m, v in monthly_rmse.items():
        bar = '█' * int(v / 5)
        print(f"    Mois {m:2d}: {v:6.1f}  {bar}")

---
## 19. Quick Feature Importance (Mutual Information)

In [ ]:
from sklearn.feature_selection import mutual_info_regression

# Préparer les features numériques (drop les datetimes et les targets)
feat_cols = [c for c in train.columns 
             if c not in ['datetime_CET', 'datetime_UTC', 'fr_spot', 'uk_spot',
                          'fr_residual', 'uk_residual', 'de_residual',
                          'fr_thermal_need', 'uk_thermal_need',
                          'spread_fr_uk', 'net_import_uk', 'fr_uk_net_flow',
                          'fr_spark', 'uk_spark',
                          'fr_wind_pen', 'uk_wind_pen', 'fr_solar_pen',
                          'fr_renewable_pen', 'uk_renewable_pen']
             and train[c].dtype in ['float64', 'int64']]

X = train[feat_cols].fillna(-999)  # MI can't handle NaN

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

for idx, target in enumerate(['fr_spot', 'uk_spot']):
    y = train[target].values
    mi = mutual_info_regression(X, y, random_state=42, n_neighbors=5)
    mi_series = pd.Series(mi, index=feat_cols).sort_values(ascending=True)
    top30 = mi_series.tail(30)
    
    axes[idx].barh(range(len(top30)), top30.values, color='steelblue')
    axes[idx].set_yticks(range(len(top30)))
    axes[idx].set_yticklabels(top30.index, fontsize=8)
    axes[idx].set_title(f'Top 30 Mutual Information — {target}')
    axes[idx].set_xlabel('MI Score')

plt.tight_layout()
plt.show()

---
## 20. Summary and recommendations for the model

In [ ]:
print("="*70)
print("SYNTHÈSE EDA — INCOMO 3")
print("="*70)

print("""
1. BASELINES
   - Naive 24h RMSE: FR ~50.65, UK ~47.23
   - Objectif: descendre significativement en dessous

2. DRIVERS CLÉS FR
   - Nucléaire (#1 driver): capacité dispo très variable (crise 2022)
   - Residual load: corrélation forte avec prix
   - Thermal need (residual - nuclear): driver direct du marginal
   - Température rivières: >25°C = contrainte nucléaire = spike
   - Gaz/carbone: coût marginal des centrales thermiques
   - Couplage continental (DE, BE, NL prix très corrélés)

3. DRIVERS CLÉS UK
   - Wind (#1 driver): UK très éolien, merit order effect fort
   - Gas capacity: UK gas-dominated quand pas de vent
   - Interconnecteurs: UK isolé du continent (Brexit)
   - Gas price UK (NBP): coût marginal direct

4. COUPLAGE FR-UK
   - Corrélation globale: ~0.92 (très forte)
   - 3 interconnecteurs FR-UK (IFA1, IFA2, ElecLink)
   - Spread moyen proche de 0 mais volatile
   - Heures de pointe = divergence maximale

5. PATTERNS TEMPORELS
   - Forte saisonnalité (hiver >> été pour les deux)
   - Profils journaliers marqués (double pic 8-9h et 18-20h)
   - Weekend ~20% moins cher
   - Spikes concentrés: août (FR), août + décembre (UK)

6. DISTRIBUTION SHIFT TRAIN → TEST
   - Train inclut crise 2022 (prix très élevés)
   - Test = période plus calme (2024-2025)
   - Nucléaire FR: en hausse dans le test (recovery)
   - Prix laggés: nettement plus bas dans le test

7. RISQUES RMSE
   - 1% des heures (spikes) peut dominer le RMSE
   - FR spikes = nucléaire bas + vent faible + été (chaleur)
   - UK spikes = vent faible + load haut + décembre
   - Huber loss ou MAE pendant le training peut aider

8. FEATURES À CRÉER
   - Residual load (FR, UK, DE)
   - Thermal need (residual - nuclear)
   - Wind/solar penetration
   - Spark spread (gas/eff + emission)
   - Spread FR-UK (lagged)
   - Interconnector net flow total
   - River temperature thresholds
   - Calendar (hour, dow, month, holiday, weekend)
   - Rolling stats (mean, std sur 24h, 168h)
""")